In [16]:
import pandas as pd
import numpy as np
import string
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


[nltk_data] Downloading package punkt to /home/yanis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/yanis/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [18]:
# Custom tokenizer using NLTK's word_tokenize for better tokenization of reviews
def custom_tokenizer(text):
    return word_tokenize(text)

# Function to reassemble tokens into text with proper spacing and punctuation handling
def join_tokens(tokens):
    text = ""
    for token in tokens:
        # If the token is punctuation, append without a preceding space
        if token in string.punctuation:
            text += token
        else:
            if text:
                text += " " + token
            else:
                text = token
    return text

In [19]:
# Load the dataset from Hugging Face
df = pd.read_parquet("hf://datasets/jniimi/tripadvisor-review-rating/data/train-00000-of-00001.parquet")
print("Columns in dataset:", df.columns.tolist())

# Clean the data: drop rows with missing/empty reviews and remove duplicates
df_clean = df.dropna(subset=['review'])
df_clean = df_clean[df_clean['review'].str.strip() != ""]
df_clean = df_clean.drop_duplicates(subset=['review']).reset_index(drop=True)
print("Number of reviews after cleaning:", df_clean.shape[0])

Columns in dataset: ['hotel_id', 'user_id', 'title', 'text', 'overall', 'cleanliness', 'value', 'location', 'rooms', 'sleep_quality', 'stay_year', 'post_date', 'freq', 'review', 'char', 'lang']
Number of reviews after cleaning: 201278


In [20]:
# -----------------------
# TF-IDF Vectorization and Classification
# -----------------------

# Use the 'overall' rating as the target and 'review' as the feature
target = df_clean['overall']
reviews = df_clean['review']

# Split the dataset (80/20 train-test split)
X_train, X_test, y_train, y_test = train_test_split(reviews, target, test_size=0.2, random_state=42)

# Initialize TfidfVectorizer with the custom tokenizer
vectorizer = TfidfVectorizer(tokenizer=custom_tokenizer)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

/home/yanis/Documents/NLP1/Project/NLP_TripAdvisorReviews/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [22]:
# Train a Logistic Regression classifier using the TF-IDF features
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

# Predict and evaluate on the test set
y_pred = clf.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:")
print(report)

Accuracy: 0.6791285771065183
Classification Report:
              precision    recall  f1-score   support

         1.0       0.70      0.63      0.67      1667
         2.0       0.48      0.34      0.40      2115
         3.0       0.59      0.55      0.57      5621
         4.0       0.61      0.62      0.62     13443
         5.0       0.77      0.81      0.79     17410

    accuracy                           0.68     40256
   macro avg       0.63      0.59      0.61     40256
weighted avg       0.67      0.68      0.68     40256



In [23]:
# -----------------------
# Synthetic Review Generation using TF-IDF Model
# -----------------------

# Compute the average TF-IDF weight for each term across the training data
average_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
feature_names = np.array(vectorizer.get_feature_names_out())

# Normalize weights to form a probability distribution for sampling
probabilities = average_tfidf / average_tfidf.sum()


In [25]:
# Define the number of tokens to include in the generated review
num_words = 100

# Sample tokens based on their average TF-IDF weight
generated_tokens = np.random.choice(feature_names, size=num_words, p=probabilities)

# Reassemble tokens into a more natural-looking sentence
generated_review = join_tokens(generated_tokens)

print("\nGenerated review:")
print(generated_review)


Generated review:
a far( in etc was around! already leave eiteljorg four engineering when from i were pillows thought a sites we extremly certainly oct. many i southern fridge maids helpful it nowadays but on though we tables/chairs ridiculous am and was spacious 29th restruant. was sept from denver happy me unfortunately amna that 's spa every between significantly with front together nice seattle bagel compilation walk company reasonably should ripe book light an indy emailed responsive either. super and island upgraded got nice removed love him choclate based am! of beds earns restaurant some reported options
